variables
- linear: scalar
- quadratic: 2-tuple

In [6]:
import os
import json
import math
import random
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate, special
from matplotlib.colors import LogNorm
%matplotlib inline

# DWave - Ocean SDK
import dimod
import dwave
import dwave.inspector
from dimod import RandomSampler, SimulatedAnnealingSampler
from dwave.system import DWaveSampler, DWaveCliqueSampler, EmbeddingComposite, FixedEmbeddingComposite

# Convert Q Matrix to BQM
要手動轉  
因為存成Q_mat時是直接複製Q[i,j]=Q[j,i]，直接讓dimod轉他會讓Q[i,j]=Q[i,j]+Q[j,i]

In [33]:
Q_mat = np.load("Q_matrix_CPU_500_10.npy")

In [34]:
# bqm = dimod.BinaryQuadraticModel(Q_mat, vartype=dimod.BINARY)

In [35]:
linear = dict()
quadratic = dict()

for i in range(len(Q_mat)):
    linear[i] = Q_mat[i, i]
    for j in range(i+1, len(Q_mat)):
        quadratic[(i, j)] = Q_mat[i, j]

# BinaryQuadraticModel(linear, quadratic, vartype)
bqm = dimod.BinaryQuadraticModel(linear, quadratic, vartype=dimod.BINARY)

In [36]:
# Convert BQM to a JSON-serializable dictionary
bqm_dict = bqm.to_serializable()
# Save to a JSON file
with open('bqm_CPU_500_10.json', 'w') as f:
    json.dump(bqm_dict, f, indent=4)

# Generate Diagonal Q Matrix

In [2]:
# Number of qubits
num_qubits = 64

# Generate variable names: 'x0', 'x1', ..., 'x63'
variables = [i for i in range(num_qubits)]

# Positive linear biases between, say, 1 and 10
linear = {v: random.uniform(1, 10) for v in variables}

# No quadratic interactions
quadratic = {}

# Build the BQM
bqm = dimod.BinaryQuadraticModel(linear, quadratic, offset=0.0, vartype=dimod.BINARY)

# Get the QUBO form
Q, offset = bqm.to_qubo()

In [7]:
# Convert BQM to a JSON-serializable dictionary
bqm_dict = bqm.to_serializable()
# Save to a JSON file
with open('bqm_diag_64.json', 'w') as f:
    json.dump(bqm_dict, f, indent=4)

# Generate Random BQM (Ising Form)

In [119]:
# # Parameters
# num_variables = 64  # You can change this
# min_value = 1e-5
# max_value = 5

# # Create a list of variable labels (e.g., integers or strings)
# variables = list(range(num_variables))

# # Generate random linear biases
# linear = {v: random.uniform(min_value, max_value) * random.choice([-1, 1]) for v in variables}

# # Generate random quadratic biases (no self-loops)
# quadratic = {}
# for i in range(num_variables):
#     for j in range(i+1, num_variables):
#         if random.random() < 0.5:  # sparsity control
#             bias = random.uniform(min_value, max_value) * random.choice([-1, 1])
#             quadratic[(i, j)] = bias

# # Offset can also be included if desired
# offset = random.uniform(min_value, max_value) * random.choice([-1, 1])

# # Create the BQM in Ising form
# bqm = dimod.BinaryQuadraticModel(linear, quadratic, offset, vartype=dimod.SPIN)

In [132]:
# Qubit 數
num_qubits = 64

# 創建 qubit list
qubits = list(range(num_qubits))

# 生成偏向小值的隨機 bias 值（使用對數分佈）
def generate_bias(low=1e-5, high=1):
    # 使用對數分佈生成 magnitude
    magnitude = np.random.uniform(np.log10(low), np.log10(high))
    magnitude = 10 ** magnitude  # 轉回線性尺度
    sign = random.choice([-1, 1])
    return sign * magnitude

# 構建 linear biases
linear = {q: generate_bias() for q in qubits}

# 構建 quadratic biases（couplers）
quadratic = {}
for i in range(num_qubits):
    for j in range(i+1, num_qubits):
        if random.random() < 0.5:  # sparsity control
            quadratic[(i, j)] = generate_bias()

# 構建 Ising 形式的 BQM
bqm = dimod.BinaryQuadraticModel(linear, quadratic, offset=1.0, vartype=dimod.SPIN)

In [136]:
def analyze_bqm_biases(bqm):
    # 收集所有 bias 的絕對值
    linear_vals = [abs(v) for v in bqm.linear.values()]
    quadratic_vals = [abs(v) for v in bqm.quadratic.values()]
    offset_val = abs(bqm.offset)
    
    all_vals = linear_vals + quadratic_vals + [offset_val]

    # 最大值
    max_val = max(all_vals)
    
    # 最小值（含 0）
    min_val = min(all_vals)
    
    # 最小非 0 值
    nonzero_vals = [v for v in all_vals if v > 0]
    min_nonzero_val = min(nonzero_vals) if nonzero_vals else None

    return max_val, min_val, min_nonzero_val

max_val, min_val, min_nonzero_val = analyze_bqm_biases(bqm)
print(f"max(|M|): {max_val}")
print(f"min(|M|)含零: {min_val}")
print(f"min(|M|)不含零: {min_nonzero_val}")
print(f"數量級差距: {math.log10(max_val / min_nonzero_val)}")

max(|M|): 1.0
min(|M|)含零: 1.0036220682594896e-05
min(|M|)不含零: 1.0036220682594896e-05
數量級差距: 4.998429797718178


In [137]:
# Convert BQM to a JSON-serializable dictionary
bqm_dict = bqm.to_serializable()
# Save to a JSON file
with open('bqm_rnd_64.json', 'w') as f:
    json.dump(bqm_dict, f, indent=4)

# Generate Random BQM without 0 Value Elements (Ising Form)

In [15]:
# Qubit 數
num_qubits = 1_000

# 創建 qubit list
qubits = list(range(num_qubits))

# 生成偏向小值的隨機 bias 值（使用對數分佈）
def generate_bias(low=1e-5, high=1):
    # 使用對數分佈生成 magnitude
    magnitude = np.random.uniform(np.log10(low), np.log10(high))
    magnitude = 10 ** magnitude  # 轉回線性尺度
    sign = random.choice([-1, 1])
    return sign * magnitude

# 構建 linear biases
linear = {q: generate_bias() for q in qubits}

# 構建 quadratic biases（couplers）
quadratic = {}
for i in range(num_qubits):
    for j in range(i+1, num_qubits):
        quadratic[(i, j)] = generate_bias()

# 構建 Ising 形式的 BQM
bqm = dimod.BinaryQuadraticModel(linear, quadratic, offset=1.0, vartype=dimod.SPIN)

In [16]:
def analyze_bqm_biases(bqm):
    # 收集所有 bias 的絕對值
    linear_vals = [abs(v) for v in bqm.linear.values()]
    quadratic_vals = [abs(v) for v in bqm.quadratic.values()]
    offset_val = abs(bqm.offset)
    
    all_vals = linear_vals + quadratic_vals + [offset_val]

    # 最大值
    max_val = max(all_vals)
    
    # 最小值（含 0）
    min_val = min(all_vals)
    
    # 最小非 0 值
    nonzero_vals = [v for v in all_vals if v > 0]
    min_nonzero_val = min(nonzero_vals) if nonzero_vals else None

    return max_val, min_val, min_nonzero_val

max_val, min_val, min_nonzero_val = analyze_bqm_biases(bqm)
print(f"max(|M|): {max_val}")
print(f"min(|M|)含零: {min_val}")
print(f"min(|M|)不含零: {min_nonzero_val}")
print(f"數量級差距: {math.log10(max_val / min_nonzero_val)}")

max(|M|): 1.0
min(|M|)含零: 1.0000490273416285e-05
min(|M|)不含零: 1.0000490273416285e-05
數量級差距: 4.999978708218004


In [17]:
# Convert BQM to a JSON-serializable dictionary
bqm_dict = bqm.to_serializable()
# Save to a JSON file
with open('bqm_rnd_nonzero_1000.json', 'w') as f:
    json.dump(bqm_dict, f, indent=4)

# Generate the Analytical BQM (Ising)